### ЗАДАЧА: Панель претензий к поставщикам по недопоставке по паттерну `MVC`

Команда procurement operations разбирает кейсы по недопоставке товара от поставщиков.
После приемки поставки система сравнивает ожидаемое количество и фактически полученный товар.
Если есть расхождение, создается кейс, по которому нужно провести расследование, запросить документы,
согласовать компенсацию от поставщика и закрыть кейс.

Нужно реализовать внутреннюю консольную панель по паттерну `MVC`, где:
- `Model` хранит кейсы и бизнес-правила;
- `View` отвечает только за вывод данных;
- `Controller` принимает действия и связывает `Model` и `View`.

## Что должно храниться в кейсе

Для каждого кейса нужно хранить:
- `case_id` — идентификатор кейса;
- `supplier` — поставщик;
- `shipment_id` — идентификатор поставки;
- `sku` — товар;
- `expected_qty` — ожидаемое количество;
- `received_qty` — фактически принятое количество;
- `unit_cost` — себестоимость единицы товара;
- `claim_qty` — количество товара, которое считается недопоставленным;
- `claim_amount` — сумма претензии к поставщику;
- `approved_compensation` — согласованная компенсация;
- `remaining_loss` — остаток убытка после компенсации;
- `status` — текущий статус кейса;
- `manager` — сотрудник, который ведет кейс;
- `documents_verified` — подтверждены ли документы;
- `decision` — итоговое решение.

## Формулы

При создании кейса и после любого изменения компенсации нужно правильно считать:
- `claim_qty = expected_qty - received_qty`
- если `claim_qty < 0`, нужно выбрасывать ошибку;
- `claim_amount = claim_qty * unit_cost`
- `remaining_loss = claim_amount - approved_compensation`
- все денежные значения нужно округлять до 2 знаков.

## Статусы кейса

- `new`
- `investigating`
- `documents_requested`
- `documents_verified`
- `ready_for_resolution`
- `fully_compensated`
- `partial_compensated`
- `rejected`
- `escalated`

## Бизнес-правила

- нельзя создать кейс с уже существующим `case_id`;
- нельзя назначить `manager` несуществующему кейсу;
- финальные кейсы (`fully_compensated`, `partial_compensated`, `rejected`, `escalated`) нельзя менять дальше;
- начать расследование можно только из `new` и только если назначен `manager`;
- запросить документы можно только из `investigating`;
- подтвердить документы можно только из `documents_requested`;
- при подтверждении документов поле `documents_verified` должно стать `True`, а статус — `documents_verified`;
- установить `approved_compensation` можно только из `investigating` или `documents_verified`;
- `approved_compensation` не может быть меньше `0`;
- `approved_compensation` не может быть больше `claim_amount`;
- после изменения `approved_compensation` нужно пересчитать `remaining_loss`;
- перевод в `ready_for_resolution` возможен только из `investigating` или `documents_verified`;
- перевод в `ready_for_resolution` невозможен, если `approved_compensation == 0` и при этом `documents_verified == False`;
- завершить кейс как `fully_compensated` можно только из `ready_for_resolution`, если `approved_compensation == claim_amount`;
- завершить кейс как `partial_compensated` можно только из `ready_for_resolution`, если `0 < approved_compensation < claim_amount`;
- завершить кейс как `rejected` можно только из `ready_for_resolution`, если `approved_compensation == 0`;
- эскалировать кейс можно только из `investigating`, `documents_verified` или `ready_for_resolution`;
- при любом финальном статусе нужно записывать `decision`.

## Что должен уметь `Model`

Нужно самостоятельно спроектировать модель, но она должна уметь минимум:
- создавать кейс;
- назначать менеджера;
- начинать расследование;
- запрашивать документы;
- подтверждать документы;
- устанавливать `approved_compensation`;
- переводить кейс в `ready_for_resolution`;
- завершать кейс как `fully_compensated`;
- завершать кейс как `partial_compensated`;
- завершать кейс как `rejected`;
- эскалировать кейс;
- возвращать список кейсов;
- возвращать summary.

## Что должен уметь `View`

Нужно реализовать вывод:
- списка кейсов;
- summary;
- успешных сообщений;
- ошибок.

Если список кейсов пустой, вывести отдельное сообщение.

## Что должен делать `Controller`

Контроллер должен:
- вызывать методы модели;
- оборачивать операции в `try/except` с `ValueError`;
- передавать результат во view;
- обработать все действия из `actions`.

## Формат строки кейса

Каждый кейс можно вывести строкой такого вида:

`case_id | supplier | shipment_id | sku | expected_qty | received_qty | claim_qty | claim_amount | approved_compensation | remaining_loss | status | manager | documents_verified | decision`

## Что должно быть в summary

Нужно вернуть словарь, в котором есть:
- количество кейсов по статусам;
- `total_claim_amount` — общая сумма претензий;
- `total_approved_compensation` — общая согласованная компенсация;
- `total_remaining_loss` — общий остаток убытка;
- `verified_docs_cases` — количество кейсов, где документы подтверждены;
- `fully_compensated_amount` — сумма компенсаций по кейсам со статусом `fully_compensated`.

## Что нужно сделать в конце

1. Создать модель, view и controller.
2. Загрузить данные из `initial_cases`.
3. Обработать все действия из `actions`.
4. В конце вывести финальное состояние кейсов и summary.

In [3]:
from dataclasses import dataclass
from typing import  Optional

initial_cases = [
    ("SC-100", "alpha-supply", "SHIP-7001", "SKU-100", 120, 102, 35.0),
    ("SC-101", "beta-distribution", "SHIP-7002", "SKU-200", 80, 80, 50.0),
]

actions = [
    ("show",),
    ("investigate", "SC-100"),
    ("assign", "SC-100", "Olga"),
    ("investigate", "SC-100"),
    ("docs_request", "SC-100"),
    ("docs_verify", "SC-100"),
    ("set_compensation", "SC-100", 500.0),
    ("ready", "SC-100"),
    ("partial", "SC-100", "supplier_accepted_partial_claim"),
    ("create", "SC-102", "gamma-warehouses", "SHIP-7003", "SKU-300", 60, 40, 28.0),
    ("assign", "SC-102", "Max"),
    ("investigate", "SC-102"),
    ("set_compensation", "SC-102", 560.0),
    ("ready", "SC-102"),
    ("full", "SC-102", "full_compensation_approved"),
    ("create", "SC-103", "delta-trade", "SHIP-7004", "SKU-400", 45, 30, 42.0),
    ("assign", "SC-103", "Ina"),
    ("investigate", "SC-103"),
    ("escalate", "SC-103", "supplier_disputes_shortage"),
    ("show",),
]

@dataclass
class ClaimCase:
    case_id: str
    supplier: str
    shipment_id: str
    sku: str
    expected_qty: int
    received_qty: int
    unit_cost: float
    claim_qty: int = 0
    approved_compensation: float = 0.0
    remaining_loss: float = 0.0
    status: str = "new"
    manager: Optional[str] = None
    documents_verified: bool = False
    decision: Optional[str] = None
    
    def __post_init__(self):
        self.calculate_values()
        
    def calculate_values(self) -> None:
        self.claim_qty = self.expected_qty - self.received_qty
        if self.claim_qty < 0:
            raise ValueError("Claim quantity cannot be negative")
        self.claim_amount = round(self.claim_qty * self.unit_cost, 2)
        self.remaining_loss = round(self.claim_amount - self.approved_compensation, 2)
        
class ClaimModel:
    def __init__(self):
        self.cases: dict[str, ClaimCase] = {}
        
    def create_case(self, case_id: str, supplier: str, shipment_id: str, sku: str,
                   expected_qty: int, received_qty: int, unit_cost: float) -> None:
        if case_id in self.cases:
            raise ValueError(f"Case with ID {case_id} already exists")
        case = ClaimCase(case_id, supplier, shipment_id, sku, expected_qty, received_qty, unit_cost)
        self.cases[case_id] = case
        
    def assign_manager(self, case_id: str, manager: str) -> None:
        if case_id not in self.cases:
            raise ValueError(f"Case {case_id} not found")
        case = self.cases[case_id]
        final_statuses = {"fully_compensated", "partial_compensated", "rejected", "escalated"}
        if case.status in final_statuses:
            raise ValueError(f"Final cases cannot be modified")
        case.manager = manager
        
    def start_investigation(self, case_id: str) -> None:
        if case_id not in self.cases:
            raise ValueError(f"Case {case_id} not found")
        case = self.cases[case_id]
        if case.status != "new":
            raise ValueError("Investigation can only be started from new status")
        if not case.manager:
            raise ValueError("Manager must be assigned before investigation")
        case.status = "investigating"
        
    def request_documents(self, case_id: str) -> None:
        if case_id not in self.cases:
            raise ValueError(f"Case {case_id} not found")
        case = self.cases[case_id]
        if case.status != "investigating":
            raise ValueError("Documents can only be requested from investigating status")
        case.status = "documents_requested"
        
    def verify_documents(self, case_id: str) -> None:
        if case_id not in self.cases:
            raise ValueError(f"Case {case_id} not found")
        case = self.cases[case_id]
        if case.status != "documents_requested":
            raise ValueError("Documents can only be verified from documents_requested status")
        case.documents_verified = True
        case.status = "documents_verified"
        
    def set_approved_compensation(self, case_id: str, compensation: float) -> None:
        if case_id not in self.cases:
            raise ValueError(f"Case {case_id} not found")
        case = self.cases[case_id]
        valid_statuses = {"investigating", "documents_verified"}
        if case.status not in valid_statuses:
            raise ValueError("Approved compensation can only be set from investigating or documents_verified status")
        if compensation < 0:
            raise ValueError("Approved compensation cannot be negative")
        if compensation > case.claim_amount:
            raise ValueError("Approved compensation cannot exceed claim amount")
        case.approved_compensation = compensation
        case.calculate_values()
        
    def mark_ready_for_resolution(self, case_id: str) -> None:
        if case_id not in self.cases:
            raise ValueError(f"Case {case_id} not found")
        case = self.cases[case_id]
        valid_statuses = {"investigating", "documents_verified"}
        if case.status not in valid_statuses:
            raise ValueError("Ready for resolution can only be marked from investigating or documents_verified status")
        if case.approved_compensation == 0 and not case.documents_verified:
            raise ValueError("Cannot mark ready for resolution with zero compensation and unverified documents")
        case.status = "ready_for_resolution"
        
    def fully_compensate(self, case_id: str, decision: str) -> None:
        if case_id not in self.cases:
            raise ValueError(f"Case {case_id} not found")
        case = self.cases[case_id]
        if case.status != "ready_for_resolution":
            raise ValueError("Case can only be fully compensated from ready_for_resolution status")
        if case.approved_compensation != case.claim_amount:
            raise ValueError("Approved compensation must equal claim amount for full compensation")
        case.status = "fully_compensated"
        case.decision = decision
        
    def partially_compensate(self, case_id: str, decision: str) -> None:
        if case_id not in self.cases:
            raise ValueError(f"Case {case_id} not found")
        case = self.cases[case_id]
        if case.status != "ready_for_resolution":
            raise ValueError("Case can only be partially compensated from ready_for_resolution status")
        if not (0 < case.approved_compensation < case.claim_amount):
            raise ValueError("Approved compensation must be between 0 and claim amount for partial compensation")
        case.status = "partial_compensated"
        case.decision = decision
        
    def reject_case(self, case_id: str, decision: str) -> None:
        if case_id not in self.cases:
            raise ValueError(f"Case {case_id} not found")
        case = self.cases[case_id]
        if case.status != "ready_for_resolution":
            raise ValueError("Case can only be rejected from ready_for_resolution status")
        if case.approved_compensation != 0:
            raise ValueError("Compensation must be 0 to reject case")
        case.status = "rejected"
        case.decision = decision
        
    def escalate_case(self, case_id: str, decision: str) -> None:
        if case_id not in self.cases:
            raise ValueError(f"Case {case_id} not found")
        case = self.cases[case_id]
        valid_statuses = {"investigating", "documents_verified", "ready_for_resolution"}
        if case.status not in valid_statuses:
            raise ValueError("Case can only be escalated from valid intermediate statuses")
        case.status = "escalated"
        case.decision = decision
        
    def list_cases(self) -> list[str]:
        if not self.cases:
            return["No cases available"]
        rows = []
        for case in self.cases.values():
            manager = case.manager if case.manager is not None else "None"
            decision = case.decision if case.decision is not None else "None"
            rows.append (
                f"{case.case_id} | {case.supplier} | {case.shipment_id} | "
                f"{case.sku} | {case.expected_qty} | {case.received_qty} | "
                f"{case.claim_qty} | {case.claim_amount} | {case.approved_compensation} | "
                f"{case.remaining_loss} | {case.status} | {manager} | "
                f"{case.documents_verified} | {decision}"
            )
        return rows
    
    def get_summary(self) -> dict:
        summary = {
            "total_claim_amount": 0.0,
            "total_approved_compensation": 0.0,
            "total_remaining_loss": 0.0,
            "verified_docs_cases": 0,
            "fully_compensated_amount": 0.0
        }
        status_counts = {}
        
        for case in self.cases.values():
            status_counts[case.status] = status_counts.get(case.status, 0) + 1
            summary["total_claim_amount"] += case.claim_amount
            summary["total_approved_compensation"] += case.approved_compensation
            summary["total_remaining_loss"] += case.remaining_loss
            if case.documents_verified:
                summary["verified_docs_cases"] += 1
            if case.status == "fully_compensated":
                summary["fully_compensated_amount"] += case.approved_compensation
                
                
                
        all_statuses = [
        "new", "investigating", "documents_requested", "documents_verified","ready_for_resolution", "fully_compensated", "partial_compensated", "rejected", "escalated"
    ]
        for status in all_statuses:
            summary[status] = status_counts.get(status, 0)

        return summary
    
class ClaimView:
        @staticmethod
        def render_cases(cases: list[str]) -> None:
            if cases == ["No cases available"]:
                print("No cases available")
            else:
                for case in cases:
                    print(case)
                    
        @staticmethod
        def render_summary(summary: dict) -> None:
            print("\nSummary:")
            for key, value in summary.items():
                print(f"{key}: {value}")
                
        @staticmethod
        def render_success(message: str) -> None:
            print(f"SUCCESS: {message}")
            
        @staticmethod
        def render_error(message: str) -> None:
            print(f"ERROR: {message}")
            
class ClaimController:
    def __init__(self, model: ClaimModel, view: ClaimView):
        self.model = model
        self.view = view
        
    def show_cases(self) -> None:
        try:
            cases = self.model.list_cases()
            self.view.render_cases(cases)
        except Exception as e:
            self.view.render_error(str(e))
            
    def create_case(self, case_id: str, supplier: str, shipment_id: str, sku: str,
                     expected_qty: int, received_qty: int, unit_cost: float) -> None:
        try:
            self.model.create_case(case_id, supplier, shipment_id, sku,expected_qty, received_qty, unit_cost)
            self.view.render_success(f"Case {case_id} created")
        except ValueError as e:
            self.view.render_error(str(e))
            
    def assign_manager(self, case_id: str, manager: str) -> None:
        try:
            self.model.assign_manager(case_id, manager)
            self.view.render_success(f"Manager assigned to {case_id}")
        except ValueError as e:
            self.view.render_error(str(e))
            
    def start_investigation(self, case_id: str) -> None:
        try:
            self.model.start_investigation(case_id)
            self.view.render_success(f"Investigation started for {case_id}")
        except ValueError as e:
            self.view.render_error(str(e))
            
    def request_documents(self, case_id: str) -> None:
        try:
            self.model.request_documents(case_id)
            self.view.render_success(f"Documents requested for {case_id}")
        except ValueError as e:
            self.view.render_error(str(e))
            
    def verify_documents(self, case_id: str) -> None:
        try:
            self.model.verify_documents(case_id)
            self.view.render_success(f"Documents verified for {case_id}")
        except ValueError as e:
            self.view.render_error(str(e))
            
    def set_compensation(self, case_id: str, compensation: float) -> None:
        try:
            self.model.set_approved_compensation(case_id, compensation)
            self.view.render_success(f"Approved compensation set for {case_id}: {compensation}")
        except ValueError as e:
            self.view.render_error(str(e))
            
    def mark_ready(self, case_id: str) -> None:
        try:
            self.model.mark_ready_for_resolution(case_id)
            self.view.render_success(f"Case {case_id} marked ready for resolution")
        except ValueError as e:
            self.view.render_error(str(e))
            
    def fully_compensate(self, case_id: str, decision: str) -> None:
        try:
            self.model.fully_compensate(case_id, decision)
            self.view.render_success(f"Case {case_id} fully compensated")
        except ValueError as e:
            self.view.render_error(str(e))
            
    def partially_compensate(self, case_id: str, decision: str) -> None:
        try:
            self.model.partially_compensate(case_id, decision)
            self.view.render_success(f"Case {case_id} partially compensated")
        except ValueError as e:
            self.view.render_error(str(e))
            
    def reject_case(self, case_id: str, decision: str) -> None:
        try:
            self.model.reject_case(case_id, decision)
            self.view.render_success(f"Case {case_id} rejected")
        except ValueError as e:
            self.view.render_error(str(e))
            
    def escalate_case(self, case_id: str, decision: str) -> None:
        try:
            self.model.escalate_case(case_id, decision)
            self.view.render_success(f"Case {case_id} escalated")
        except ValueError as e:
            self.view.render_error(str(e))
            
    def get_summary(self) -> dict:
         summary = self.model.get_summary()
         self.view.render_summary(summary)
         return summary
        

model = ClaimModel()
view = ClaimView()
controller = ClaimController(model, view)


for case_data in initial_cases:
    controller.create_case(*case_data)
    
for action in actions:
    action_type = action[0]
    try:
        if action_type == "show":
            controller.show_cases()
        elif action_type == "create":
            controller.create_case(*action[1:])
        elif action_type == "assign":
            controller.assign_manager(action[1], action[2])
        elif action_type == "investigate":
            controller.start_investigation(action[1])
        elif action_type == "docs_request":
            controller.request_documents(action[1])
        elif action_type == "docs_verify":
            controller.verify_documents(action[1])
        elif action_type == "set_compensation":
            controller.set_compensation(action[1], action[2])
        elif action_type == "ready":
            controller.mark_ready(action[1])
        elif action_type == "full":
            controller.fully_compensate(action[1], action[2])
        elif action_type == "partial":
            controller.partially_compensate(action[1], action[2])
        elif action_type == "reject":
            controller.reject_case(action[1], action[2])    
        elif action_type == "escalate":
            controller.escalate_case(action[1], action[2])
    except Exception as e:
        controller.view.render_error(f"Action {action} failed: {str(e)}")     
        
print("Финальное состояние:")
controller.show_cases()    

print("Сводка по всем кейсам:")
controller.get_summary()                      
                    
        
        
            
            
            
        
            
        
        
        
        
        
        
        
        
        

SUCCESS: Case SC-100 created
SUCCESS: Case SC-101 created
SC-100 | alpha-supply | SHIP-7001 | SKU-100 | 120 | 102 | 18 | 630.0 | 0.0 | 630.0 | new | None | False | None
SC-101 | beta-distribution | SHIP-7002 | SKU-200 | 80 | 80 | 0 | 0.0 | 0.0 | 0.0 | new | None | False | None
ERROR: Manager must be assigned before investigation
SUCCESS: Manager assigned to SC-100
SUCCESS: Investigation started for SC-100
SUCCESS: Documents requested for SC-100
SUCCESS: Documents verified for SC-100
SUCCESS: Approved compensation set for SC-100: 500.0
SUCCESS: Case SC-100 marked ready for resolution
SUCCESS: Case SC-100 partially compensated
SUCCESS: Case SC-102 created
SUCCESS: Manager assigned to SC-102
SUCCESS: Investigation started for SC-102
SUCCESS: Approved compensation set for SC-102: 560.0
SUCCESS: Case SC-102 marked ready for resolution
SUCCESS: Case SC-102 fully compensated
SUCCESS: Case SC-103 created
SUCCESS: Manager assigned to SC-103
SUCCESS: Investigation started for SC-103
SUCCESS: Cas

{'total_claim_amount': 1820.0,
 'total_approved_compensation': 1060.0,
 'total_remaining_loss': 760.0,
 'verified_docs_cases': 1,
 'fully_compensated_amount': 560.0,
 'new': 1,
 'investigating': 0,
 'documents_requested': 0,
 'documents_verified': 0,
 'ready_for_resolution': 0,
 'fully_compensated': 1,
 'partial_compensated': 1,
 'rejected': 0,
 'escalated': 1}